In [ ]:
# https://colab.research.google.com/drive/1nzdTOey1S8aHnFUxaRc-8d9zI8cEpo5R#scrollTo=FtViIAfTizDJ

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout
from tensorflow.keras.layers import concatenate

hidden_dim = 2
num_classes = 10
batch_size = 100  # должно быть кратно 60 000 и 10 0000

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)


def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))

input_img = Input(shape=(28, 28, 1))
fl = Flatten()(input_img)
lb = Input(shape=(num_classes,))
x = concatenate([fl, lb])
x = Dense(256, activation='relu')(x)
x = dropout_and_batch(x)
x = Dense(128, activation='relu')(x)
x = dropout_and_batch(x)

z_mean2 = Dense(hidden_dim)(x)
z_log_var = Dense(hidden_dim)(x)

def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean2, z_log_var])

input_dec = Input(shape=(hidden_dim,))
lb_dec = Input(shape=(num_classes,))
d = concatenate([input_dec, lb_dec])
d = Dense(128, activation='elu')(d)
d = dropout_and_batch(d)
d = Dense(256, activation='elu')(d)
d = dropout_and_batch(d)
d = Dense(28*28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

encoder = keras.Model([input_img, lb], h, name='encoder')
decoder = keras.Model([input_dec, lb_dec], decoded, name='decoder')
cvae = keras.Model([input_img, lb, lb_dec], decoder([encoder([input_img, lb]), lb_dec]), name="cvae")

z_meaner = keras.Model([input_img, lb], z_mean2)
tr_style = keras.Model([input_img, lb, lb_dec], decoder([z_meaner([input_img, lb]), lb_dec]), name='tr_style')

def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return (loss + kl_loss)/2/28/28

cvae.compile(optimizer='adam', loss=vae_loss)

cvae.fit([x_train, y_train_cat, y_train_cat], x_train, epochs=5, batch_size=batch_size, shuffle=True)

lb = lb_dec = y_test_cat
h = encoder.predict([x_test, lb], batch_size=batch_size)
plt.scatter(h[:, 0], h[:, 1])


n = 4
total = 2*n+1
input_lbl = np.zeros((1, num_classes))
input_lbl[0, 5] = 1

plt.figure(figsize=(total, total))

h = np.zeros((1, hidden_dim))
num = 1
for i in range(-n, n+1):
    for j in range(-n, n+1):
        ax = plt.subplot(total, total, num)
        num += 1
        h[0, :] = [1*i/n, 1*j/n]
        img = decoder.predict([h, input_lbl])
        plt.imshow(img.squeeze(), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)


def plot_digits(*images):
    images = [x.squeeze() for x in images]
    n = min([x.shape[0] for x in images])

    plt.figure(figsize=(n, len(images)))
    for j in range(n):
        for i in range(len(images)):
            ax = plt.subplot(len(images), n, i*n + j + 1)
            plt.imshow(images[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()

dig1 = 5
dig2 = 2

num = 10
X = x_train[y_train == dig1][:num]

lb_1 = np.zeros((num, num_classes))
lb_1[:, dig1] = 1

plot_digits(X)

for i in range(num_classes):
    lb_2 = np.zeros((num, num_classes))
    lb_2[:, i] = 1

    Y = tr_style.predict([X, lb_1, lb_2], batch_size=num)
    plot_digits(Y)

## Izah

In [ ]:
# https://colab.research.google.com/drive/1nzdTOey1S8aHnFUxaRc-8d9zI8cEpo5R#scrollTo=FtViIAfTizDJ
# (Kodun orijinal Colab ünvanı.)

# =====================================================================
# BU KOD NƏDİR? — CVAE (Conditional Variational Autoencoder,
#                       "şərtli variasiyalı avtoenkoder")
# =====================================================================
#
# Əvvəlki VAE ilə müqayisədə ƏSAS YENİLİK: şəbəkəyə ŞƏKİLLə yanaşı,
# onun ETİKETİ (hansı rəqəm olduğu, 0..9) də verilir.
#
#   VAE:   şəkil ──► encoder ──► (orta, yayılma) ──► z ──► decoder ──► şəkil
#   CVAE:  şəkil + etiket ──► encoder ──► (orta, yayılma) ──► z
#                                                     z + etiket ──► decoder ──► şəkil
#
# NİYƏ ETİKET VERİLİR? (ən vacib fikir)
#   Adi VAE-də gizli fəza (2 ədəd) HƏM "bu hansı rəqəmdir", HƏM DƏ "necə
#   yazılıb" (qalınlıq, əyiklik, ölçü) məlumatını daşımalıdır. İkisi qarışır.
#   CVAE-də "hansı rəqəm" məlumatı etiketlə HAZIR verilir. Ona görə gizli
#   fəza (z) yalnız QALAN şeyi, yəni YAZI ÜSLUBUNU (style) öyrənir.
#
#   Nəticədə iki gözəl imkan yaranır:
#     1) Rəqəmi SEÇƏ bilirik: decoder-ə etiket = 5 versək, həmişə 5 çəkir;
#        z-ni dəyişsək, yalnız 5-in üslubu (əyiklik, qalınlıq) dəyişir.
#     2) ÜSLUB KÖÇÜRMƏSİ: bir şəklin üslubunu (z) götürüb, başqa etiketlə
#        decoder-ə versək, eyni əl yazısı üslubunda BAŞQA rəqəm alırıq.
#        (Kodun sonundakı tr_style modeli məhz bunu edir.)
#
# BİRİNCİ KODLA MÜQAYİSƏ:
#
#                       VAE (1-ci kod)          CVAE (bu kod)
# Encoder girişi:       şəkil                   şəkil + etiket
# Decoder girişi:       z (2 ədəd)              z (2 ədəd) + etiket (10 ədəd)
# Gizli fəza nəyi saxlayır: rəqəm + üslub      yalnız üslub
# Gizli fəza aralığı (grid): [-3, 3]            [-1, 1]
# batch_size:           60                      100
# Loss miqyası:         cəm (onlarla)           cəm / 2 / 784 (0.0x səviyyəsi)
# Decoder aktivasiyası: relu                    elu
# Əlavə model:          yoxdur                  tr_style (üslub köçürmə)
# Etiket formatı:       yoxdur                  one-hot (aşağıda izah)
#
#   şəkil (784) ─┐
#                ├─► concatenate (794) ─► [ENCODER] ─► z_mean, z_log_var ─► h (2)
#   etiket (10) ─┘                                                          │
#                                                                           ▼
#                                                  h (2) ─┐
#                                                         ├─► concatenate (12) ─► [DECODER] ─► şəkil
#                                       etiket (10) ──────┘


# ---------------------------------------------------------------------
# 0) KİTABXANALAR
# ---------------------------------------------------------------------

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un konsola yazdığı "məlumat" və "xəbərdarlıq" mesajlarını azaldır.
#   '0' hamısı, '1' info gizlənir, '2' info + warning gizlənir, '3' yalnız xəta qalır.
# Kodun işinə təsiri yoxdur, yalnız çıxışı təmiz saxlayır.

import numpy as np
# Massivlərlə iş: np.reshape, np.zeros (etiket massivləri), np.expand_dims və s.

import matplotlib.pyplot as plt
# Qrafiklər: plt.scatter (nöqtə buludu), plt.subplot (şəkil cədvəli), plt.imshow.

import tensorflow as tf
# Xüsusi əməliyyatlar üçün lazımdır: tf.random.normal, tf.exp, tf.square,
# tf.reduce_sum, tf.reshape (noiser və vae_loss funksiyalarında).

from tensorflow.keras.datasets import mnist
# MNIST: əl ilə yazılmış rəqəmlər, 60000 öyrətmə + 10000 test, 28x28, 0..255.

from tensorflow import keras
# keras.Model və keras.utils.to_categorical üçün.

from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout
# Dense → tam bağlı qat; Flatten → (28,28,1)→(784,); Reshape → tərsi;
# Input → giriş yeri; Lambda → öz funksiyamızı qat kimi qoşmaq (noiser);
# BatchNormalization → batch üzrə normallaşdırma; Dropout → təsadüfi söndürmə.

from tensorflow.keras.layers import concatenate
# YENİ (1-ci kodda yox idi). İki vektoru yan-yana birləşdirir.
# Məsələn (784,) və (10,) → (794,). Etiketi şəkilə "yapışdırmaq" üçün lazımdır.


# ---------------------------------------------------------------------
# 1) PARAMETRLƏR
# ---------------------------------------------------------------------

hidden_dim = 2
# Gizli fəzanın ölçüsü. 2 seçilib ki, müstəvidə nöqtə kimi görünsün.

num_classes = 10
# Rəqəmlərin sayı (0..9). Etiket vektorunun uzunluğu.

batch_size = 100  # должно быть кратно 60 000 и 10 0000
# (Rusca şərh: "60 000 və 10 000-ə bölünməlidir"; "10 0000" yazı səhvidir,
#  nəzərdə tutulan 10 000-dir.)
#
# 60000 / 100 = 600 addım hər epoch-da, 10000 / 100 = 100 addım test üçün,
# qalıq YOXDUR. NİYƏ VACİBDİR? noiser və vae_loss funksiyalarında batch
# ölçüsü SABİT yazılıb (batch_size, ...). Yarımçıq batch olsa, forma uyğun
# gəlməz və xəta çıxar. Ona görə fit və predict-də də eyni batch_size verilir.


# ---------------------------------------------------------------------
# 2) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# x_train (60000, 28, 28), y_train (60000,) rəqəmlər 0..9
# x_test  (10000, 28, 28), y_test  (10000,)
# Bu dəfə y (etiketlər) İSTİFADƏ OLUNUR (1-ci kodda yox idi).

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksellər 0..255 → 0..1. Decoder-in sigmoid çıxışı da 0..1-dir, ona görə
# bərpa ilə orijinal eyni miqyasda müqayisə olunur.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))
# Kanal ox əlavə edir: (60000, 28, 28) → (60000, 28, 28, 1). Elementlərin sayı dəyişmir.

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)
# ONE-HOT KODLAŞDIRMA. Rəqəmi (məsələn 5) 10 ədədlik vektora çevirir:
#   5  →  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]   (yalnız 5-ci mövqe 1, qalanı 0)
#   2  →  [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
# NİYƏ? Şəbəkəyə "5" ədədini birbaşa versək, "5 rəqəmi 2-dən böyükdür, 4-ə
# yaxındır" kimi YANLIŞ mənalar çıxarar. One-hot isə hər sinfi bərabər və
# müstəqil göstərir.
# Forma: y_train_cat (60000, 10), y_test_cat (10000, 10).


# ---------------------------------------------------------------------
# 3) KÖMƏKÇİ FUNKSİYA
# ---------------------------------------------------------------------

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))
# x-i əvvəl BatchNormalization-dan, sonra Dropout(0.3)-dən keçirir.
# Sıra: Dense → BatchNorm → Dropout.
# BatchNorm: hər batch-də çıxışı normallaşdırır, öyrənməni sabitləşdirir.
# Dropout(0.3): öyrətmədə neyronların 30%-ni təsadüfi 0 edir (əzbərləməyə
# qarşı); predict zamanı avtomatik söndürülür.
# Hər çağırışda YENİ qatlar yaranır, qatlar paylaşılmır.


# ---------------------------------------------------------------------
# 4) ENCODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   şəkil (28,28,1) ─► Flatten (784,) ─┐
#                                      ├─► concatenate (794,) ─► Dense 256 ─► BN+Drop
#   etiket (10,) ──────────────────────┘      ─► Dense 128 ─► BN+Drop ─┬─► z_mean2    (2,)
#                                                                      └─► z_log_var  (2,)
#                                                                              │
#                                                                              ▼
#                                                              noiser (Lambda) ─► h (2,)

input_img = Input(shape=(28, 28, 1))
# 1-ci giriş: şəkil. Batch ölçüsü yazılmır.

fl = Flatten()(input_img)
# (28, 28, 1) → (784,). Parametr: yoxdur.

lb = Input(shape=(num_classes,))
# 2-ci giriş: etiket (one-hot, 10 ədəd). Encoder-in İKİ girişi var.
# DİQQƏT: lb burada simvolik giriş tensorudur. Kodun sonunda eyni ad
# (lb) numpy massivi üçün yenidən istifadə olunur.

x = concatenate([fl, lb])
# (784,) və (10,) → (794,). Şəkilin piksellərinin yanına etiket yapışdırılır.
# Beləcə encoder həm şəkli, həm onun rəqəm olduğunu görür.
# Parametr: yoxdur.

x = Dense(256, activation='relu')(x)
# 794 → 256, ReLU. Parametr: 794 × 256 + 256 = 203520.

x = dropout_and_batch(x)
# 256 ölçüsü saxlanır. BatchNorm: 4 × 256 = 1024 parametr (512-si öyrənilir).

x = Dense(128, activation='relu')(x)
# 256 → 128, ReLU. Parametr: 256 × 128 + 128 = 32896.

x = dropout_and_batch(x)
# BatchNorm: 4 × 128 = 512 parametr.

z_mean2 = Dense(hidden_dim)(x)
# 128 → 2, aktivasiya YOXDUR (linear). Parametr: 128 × 2 + 2 = 258.
# Mənası: şəklin gizli fəzadakı "buludunun" MƏRKƏZİ (2 koordinat).
# Aktivasiya qoymuruq, çünki mərkəz mənfi də, müsbət də ola bilər.
#
# NİYƏ ADI z_mean2? Çünki aşağıdakı noiser funksiyası içində "global z_mean"
# adlı ayrıca qlobal dəyişən mənimsədilir. Qatın özünü z_mean2 adlandırmaq
# onu modellərdə (z_meaner) aydın istifadə etməyə imkan verir.
# (1-ci kodda ad z_mean idi.)

z_log_var = Dense(hidden_dim)(x)
# 128 → 2, aktivasiya YOXDUR. Parametr: 258.
# Mənası: buludun yayılmasının LOQARİFMİ, log(σ²).
# NİYƏ log(σ²)? Dispersiya həmişə müsbət olmalıdır, şəbəkə isə istənilən ədəd
# çıxara bilir. σ² = exp(log σ²) həmişə müsbətdir, həm də sabitliyi artırır.

def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean
# REPARAMETRİZASİYA HİLƏSİ (1-ci kodla eynidir).
#
#   args = [z_mean2, z_log_var] (Lambda qatı belə verir).
#
#   global z_mean, z_log_var:
#     Funksiyanın içində bu iki ad GLOBAL dəyişənlərə mənimsədilir. Məqsəd:
#     aşağıdakı vae_loss funksiyası KL hissəsi üçün həmin tensorlara çata
#     bilsin. Yəni z_mean burada z_mean2 ilə EYNİ tensordur (noiser çağırılanda
#     args[0] məhz z_mean2-dir). Bu, loss-a məlumat ötürmək üçün "qısa yoldur",
#     işləyir, amma təmiz üsul deyil (gizli yan təsir). Təmiz variant: xüsusi
#     Layer yazıb add_loss istifadə etmək. Yeni Keras versiyalarında bu
#     "global" üsul xəta verə bilər; orijinal kod TF 2.x köhnə davranışına
#     güvənir.
#
#   N = tf.random.normal(...):
#     Forma (100, 2). Hər şəkil üçün 2 təsadüfi ədəd, orta 0, σ = 1.
#     Bunlar adi onluq ədədlərdir (məsələn 0.5, -1.2), normal paylanma
#     qanununa uyğun seçilir; çoxu −3..3 arasında olur, amma sərt sərhəd yoxdur.
#
#   return exp(z_log_var / 2) * N + z_mean:
#     exp(log σ² / 2) = σ.  Yəni  h = σ · N + μ.
#     Bu, "μ mərkəzli, σ yayılmalı normal paylanmadan təsadüfi nöqtə" deməkdir.
#
# NİYƏ BELƏ? Təsadüfi seçmə əməliyyatından gradient axmır. Təsadüfiliyi N-ə
# köçürürük (öyrənilən parametri yoxdur), μ və σ adi toplama/vurma ilə daxil
# olur. Beləcə gradient μ və σ-ya çatır və encoder öyrənə bilir.
#
# DİQQƏT: batch_size sabit yazıldığı üçün noiser yalnız TAM batch qəbul edir.

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean2, z_log_var])
# noiser funksiyasını şəbəkəyə qat kimi qoşur, iki giriş verir.
# h: hər şəkil üçün gizli fəzada təsadüfi seçilmiş nöqtə, forma (batch, 2).
# Parametr: yoxdur.


# ---------------------------------------------------------------------
# 5) DECODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   z (2,) ──────┐
#                ├─► concatenate (12,) ─► Dense 128 ─► BN+Drop ─► Dense 256 ─► BN+Drop
#   etiket (10,) ┘                                   ─► Dense 784 (sigmoid) ─► Reshape (28,28,1)

input_dec = Input(shape=(hidden_dim,))
# Decoder-in 1-ci girişi: gizli fəzadan 2 ədəd (üslub).

lb_dec = Input(shape=(num_classes,))
# Decoder-in 2-ci girişi: etiket (one-hot, 10 ədəd). "Hansı rəqəmi çək" əmri.
# Bu, encoder-dəki lb-dən AYRI giriş yeridir. Səbəb: üslub köçürmədə
# encoder-ə bir etiket (köhnə rəqəm), decoder-ə başqa etiket (yeni rəqəm)
# vermək lazımdır.

d = concatenate([input_dec, lb_dec])
# (2,) və (10,) → (12,). Üslub + "hansı rəqəm" birləşir.

d = Dense(128, activation='elu')(d)
# 12 → 128, ELU. Parametr: 12 × 128 + 128 = 1664.
# ELU (exponential linear unit): ReLU-ya oxşayır, amma mənfi tərəfdə kəskin
# 0 yox, hamar eksponensial əyri verir; "ölü neyron" problemini azaldır.
#   elu(x) = x            (x > 0)
#   elu(x) = e^x − 1      (x ≤ 0)

d = dropout_and_batch(d)
# BatchNorm: 4 × 128 = 512 parametr.

d = Dense(256, activation='elu')(d)
# 128 → 256, ELU. Parametr: 128 × 256 + 256 = 33024.

d = dropout_and_batch(d)
# BatchNorm: 4 × 256 = 1024 parametr.

d = Dense(28*28, activation='sigmoid')(d)
# 256 → 784, sigmoid (çıxış 0..1, piksel aralığı ilə eyni).
# Parametr: 256 × 784 + 784 = 201488.

decoded = Reshape((28, 28, 1))(d)
# (784,) → (28, 28, 1). Parametr: yoxdur.


# ---------------------------------------------------------------------
# 6) MODELLƏRİN YIĞILMASI
# ---------------------------------------------------------------------

encoder = keras.Model([input_img, lb], h, name='encoder')
# Girişlər: şəkil və etiket. Çıxış: h (təsadüfi seçilmiş 2 ədəd).
# DİQQƏT: çıxış z_mean yox, h-dır, ona görə eyni giriş iki dəfə
# predict olunsa, nəticə bir qədər fərqli olur.

decoder = keras.Model([input_dec, lb_dec], decoded, name='decoder')
# Girişlər: 2 ədəd və etiket. Çıxış: şəkil (28, 28, 1).
# Decoder-də noiser YOXDUR, ona görə eyni giriş həmişə eyni şəkli verir.

cvae = keras.Model([input_img, lb, lb_dec], decoder([encoder([input_img, lb]), lb_dec]), name="cvae")
# Bütöv model, ÜÇ giriş: şəkil, encoder üçün etiket, decoder üçün etiket.
# Axın: şəkil+etiket → encoder → h → decoder(h + etiket) → şəkil.
# encoder, decoder və cvae ÇƏKİLƏRİ PAYLAŞIR; cvae-ni öyrətdikdə hamısı öyrənir.

z_meaner = keras.Model([input_img, lb], z_mean2)
# YENİ köməkçi model: şəkil+etiket → z_mean2 (yəni yalnız MƏRKƏZ, təsadüfi
# səs-küy YOXDUR). Encoder ilə eyni qatları paylaşır (ayrıca öyrədilmir).
# Niyə lazımdır? Üslub köçürmədə təsadüfilik istəmirik: şəklin sabit,
# deterministik üslub nöqtəsi lazımdır (buludun mərkəzi).

tr_style = keras.Model([input_img, lb, lb_dec], decoder([z_meaner([input_img, lb]), lb_dec]), name='tr_style')
# ÜSLUB KÖÇÜRMƏ modeli ("transfer style"):
#   şəkil + köhnə etiket → z_meaner → üslub nöqtəsi (2 ədəd)
#   üslub nöqtəsi + YENİ etiket → decoder → yeni rəqəm, eyni üslub.
# cvae-dən fərqi: ortada noiser yox, z_meaner var (təsadüfilik yox).
# Bu model ayrıca öyrədilmir (compile/fit çağırılmır), yalnız predict üçündür,
# çünki çəkilər cvae-nin öyrədilməsi ilə artıq hazırdır.


# ---------------------------------------------------------------------
# 7) LOSS FUNKSİYASI
# ---------------------------------------------------------------------

def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return (loss + kl_loss)/2/28/28
# 1-ci kodla eynidir, bir fərqlə: sonda (loss + kl_loss) / 2 / 28 / 28.
# Burada x = orijinal (y_true), y = bərpa (y_pred); funksiyanın daxilindəki
# bu x, y adları yuxarıdakı x ilə qarışdırılmamalıdır.
#
# a) BƏRPA XƏTASI: hər şəkil üçün 784 pikselin kvadrat xətalarının CƏMİ.
# b) KL-DİVERGENSİYA: -0.5 · Σ (1 + log σ² − μ² − σ²).
#    Hər şəklin buludunu (μ, σ) standart normal N(0, 1)-dən çox
#    uzaqlaşmamağa məcbur edir (μ² mərkəzi 0-a, σ² − log σ² − 1 yayılmanı 1-ə çəkir).
#    z_mean və z_log_var burada noiser-də mənimsədilmiş GLOBAL dəyişənlərdir.
#
# SONDAKI BÖLMƏ: /2/28/28 = /1568. 784 piksel və 2 (latent ölçü) üzrə
# normallaşdırır. Nəticə: loss rəqəmləri onlarla yox, 0.0x səviyyəsində olur,
# yəni əvvəlki "ortalama MSE" miqyasına yaxınlaşır. Diqqət: hər iki hissə
# (bərpa və KL) eyni ədədə bölünür, yəni NİSBƏT dəyişmir, yalnız ümumi miqyas
# kiçilir. Optimallaşmanın istiqaməti eyni qalır, amma Adam üçün gradient
# miqyası fərqli olur.
#
# BÖYÜK FİKİR: etiket artıq encoder-ə və decoder-ə verildiyi üçün, bərpa
# xətasını azaltmaq "hansı rəqəm" məlumatını z-yə yükləmək tələb etmir.
# KL isə z-ni standart normala yaxın saxlayır. Nəticədə z yalnız üslubu daşıyır.

cvae.compile(optimizer='adam', loss=vae_loss)
# Adam optimizer, defolt öyrənmə sürəti 0.001. Loss: özümüzün vae_loss.


# ---------------------------------------------------------------------
# 8) ÖYRƏTMƏ
# ---------------------------------------------------------------------

cvae.fit([x_train, y_train_cat, y_train_cat], x_train, epochs=5, batch_size=batch_size, shuffle=True)
# Giriş SİYAHISI (3 ədəd, modelin 3 girişi ilə sırası eyni):
#   x_train       → şəkillər (input_img)
#   y_train_cat   → encoder üçün etiket (lb)
#   y_train_cat   → decoder üçün etiket (lb_dec), eyni etiket, çünki öyrətmədə
#                   şəklin öz rəqəmini çəkməyi öyrənirik
# Hədəf: x_train (avtoenkoder məntiqi: çıxış girişi bərpa etməlidir).
#
# epochs=5, batch_size=100 → hər epoch-da 600 addım, cəmi 3000 yeniləmə.
# NƏ GÖZLƏMƏK? Loss normallaşdırıldığı üçün kiçik rəqəmlər (0.0x) çıxacaq,
# epoch-lar keçdikcə azalmalıdır. Dəqiq rəqəm hər işlətmədə dəyişir.


# ---------------------------------------------------------------------
# 9) GİZLİ FƏZANIN GÖRÜNTÜSÜ
# ---------------------------------------------------------------------

lb = lb_dec = y_test_cat
# DİQQƏT: lb adı artıq simvolik Input yox, numpy massivi (10000, 10) olur.
# Modellər artıq qurulub və yadda saxlanıb, ona görə köhnə ad üzərinə
# yazmaq modellərə ZƏRƏR VERMİR, amma qarışıqlığa səbəb ola bilər.
# lb_dec isə bu kodda sonra istifadə olunmur (yalnız bir ad bəyanıdır).

h = encoder.predict([x_test, lb], batch_size=batch_size)
# Bütün 10000 test şəklini (etiketləri ilə) encoder-dən keçirir.
# Forma: (10000, 2). 10000 % 100 == 0, yarımçıq batch yoxdur.
# h SƏS-KÜYLÜ çıxışdır (z_mean yox).

plt.scatter(h[:, 0], h[:, 1])
# 10000 nöqtəni müstəvidə çəkir.
# GÖZLƏNTİ: nöqtələr 0 ətrafında bir bulud əmələ gətirir (KL-in təsiri).
# 1-ci koddan FƏRQ: orada rəqəmlər bu buludda ayrı-ayrı sahələrdə yığılırdı.
# Burada isə etiket decoder-ə ayrıca verildiyi üçün rəqəmlər BİR-BİRİNƏ
# QARIŞMIŞ olmalıdır: z rəqəmi yox, üslubu kodlayır.
# Rəngləmək üçün: plt.scatter(h[:, 0], h[:, 1], c=y_test, cmap='tab10', s=2)
# Notebook-da qrafik avtomatik görünür; .py faylda plt.show() lazımdır.


# ---------------------------------------------------------------------
# 10) SEÇİLMİŞ RƏQƏMİN ÜSLUBLARI: GİZLİ FƏZANIN ŞƏBƏKƏSİ
# ---------------------------------------------------------------------

n = 4
# Hər istiqamətdə 4 addım.

total = 2*n+1
# 2·4 + 1 = 9. Şəbəkə 9 × 9 = 81 şəkildən ibarət olacaq.

input_lbl = np.zeros((1, num_classes))
input_lbl[0, 5] = 1
# Etiket vektoru: yalnız 5-ci mövqe 1 → "5 rəqəmini çək". Forma (1, 10).
# Rəqəmi dəyişmək üçün 5 əvəzinə 0..9 yazmaq kifayətdir.

plt.figure(figsize=(total, total))
# Yeni qrafik pəncərəsi (9×9 düym). scatter ilə üst-üstə düşmür.

h = np.zeros((1, hidden_dim))
# Boş (1, 2) massiv: hər dəfə içinə gizli nöqtənin koordinatlarını yazacağıq.
# (Bu h də yuxarıdakı numpy h-ın üzərinə yazılır, ona görə scatter-dən
# sonra gəlir.)

num = 1
# plt.subplot nömrəsi 1-dən başlayır.

for i in range(-n, n+1):
    for j in range(-n, n+1):
        # i, j: -4, -3, ..., 0, ..., 4. Hər cüt bir nöqtəni təyin edir.
        ax = plt.subplot(total, total, num)
        num += 1
        h[0, :] = [1*i/n, 1*j/n]
        # Nöqtə: [i/4, j/4]. i=-4 → -1, i=0 → 0, i=4 → +1.
        # Aralıq -1..1 (addım 0.25). 1-ci kodda -3..3 idi. Burada daha dar
        # götürülüb, çünki 2 ölçülü və kiçik modeldə mərkəz ətrafı ən
        # keyfiyyətli sahədir.
        img = decoder.predict([h, input_lbl])
        # Decoder-ə iki giriş verilir: z nöqtəsi və etiket (5).
        # Nəticə: (1, 28, 28, 1).
        # Etiket HƏMİŞƏ 5 olduğu üçün hər xanada 5 görünür; yalnız z-yə görə
        # üslub dəyişir (qalınlıq, əyiklik, forma).
        plt.imshow(img.squeeze(), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

# NƏTİCƏ: 9×9 cədvəl, hamısı 5 rəqəmi, üslubu tədricən dəyişir.
# 1-ci koddan fərq: orada qonşu xanalarda rəqəmin ÖZÜ dəyişirdi (1 → 7 → 9);
# burada rəqəm sabit, ÜSLUB dəyişir. Bu CVAE-nin məqsədidir.


# ---------------------------------------------------------------------
# 11) ÜSLUB KÖÇÜRMƏ
# ---------------------------------------------------------------------

def plot_digits(*images):
    images = [x.squeeze() for x in images]
    n = min([x.shape[0] for x in images])

    plt.figure(figsize=(n, len(images)))
    for j in range(n):
        for i in range(len(images)):
            ax = plt.subplot(len(images), n, i*n + j + 1)
            plt.imshow(images[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()
# Köməkçi funksiya: bir neçə şəkil dəstini cədvəl kimi çəkir.
#   *images           → istənilən sayda dəst qəbul edir (hər dəst bir sətir olur).
#   x.squeeze()       → (N, 28, 28, 1) → (N, 28, 28).
#   n                 → dəstlərdəki ən kiçik şəkil sayı (sütun sayı).
#   i*n + j + 1       → subplot nömrəsi (sətir i, sütun j).
#   plt.gray()        → boz rəng xəritəsi.
#   plt.show()        → qrafiki göstərir (bu funksiya hər çağırışda göstərir,
#                       ona görə hər dəst ayrı şəkildə çıxır).
# Bu funksiyada x və n adları yalnız lokaldır, kodun qalan x və n-nə təsir etmir.

dig1 = 5
dig2 = 2
# dig1: üslubu götürəcəyimiz rəqəm (5). dig2 bu kodda İSTİFADƏ OLUNMUR.

num = 10
# Neçə şəkil götürürük. (Bu num əvvəlki dövrdəki num-u əvəz edir.)

X = x_train[y_train == dig1][:num]
# x_train-dən yalnız 5-ləri seçir, ilk 10-nu götürür. Forma (10, 28, 28, 1).
# y_train == dig1 → True/False massivi; indekslə seçmə ("maska").

lb_1 = np.zeros((num, num_classes))
lb_1[:, dig1] = 1
# Bu 10 şəklin ƏSL etiketi: hamısı 5. Forma (10, 10), hər sətirdə 5-ci mövqe 1.

plot_digits(X)
# Orijinal 10 beşliyi göstərir (müqayisə üçün baza sətri).

for i in range(num_classes):
    lb_2 = np.zeros((num, num_classes))
    lb_2[:, i] = 1
    # YENİ etiket: bütün şəkillər üçün "i rəqəmini çək".

    Y = tr_style.predict([X, lb_1, lb_2], batch_size=num)
    # tr_style-ın 3 girişi: şəkillər, onların köhnə etiketi (5), yeni etiket (i).
    # İş: encoder hissəsi hər beşliyin üslub nöqtəsini (z_mean) tapır,
    # decoder həmin üslubu yeni etiketlə (i) birləşdirib i rəqəmini çəkir.
    # tr_style-də noiser yoxdur, ona görə batch_size=num (10) problem yaratmır
    # və nəticə deterministikdir.

    plot_digits(Y)
    # Hər i üçün bir cədvəl: 10 şəkil, hamısı i rəqəmi, amma hər biri müvafiq
    # beşliyin əl yazısı üslubunda (əyiklik, qalınlıq, ölçü).
    # Cəmi 10 qrafik çıxır (i = 0..9).

# SON QEYDLƏR:
#   1) Üslub köçürmə "mükəmməl" olmaz: gizli fəza cəmi 2 ölçülüdür, buna görə
#      üslub məlumatı çox qabarıq yox, təxminidir. Hidden_dim artırılsa və
#      epoch-lar çoxaldılsa, keyfiyyət yaxşılaşar.
#   2) 5 epoch azdır; şəkillər bulanıq ola bilər.
#   3) batch_size dəyişsəniz, 60000 və 10000-ə bölünməsini yoxlayın.
#   4) Notebook-dan kənarda plt.show() sonda (scatter və 9×9 şəbəkədən sonra)
#      əlavə edin, çünki plot_digits yalnız öz qrafikini göstərir.

## Mənası

Yaxşı sual. Doğrudan cavab: **label-i özümüz veririksə, bu "proqnoz" deyil, generasiyadır.** Praktikada bunun mənası da məhz budur.

## Praktikada nəyə lazımdır?

**1. Sifarişlə məlumat generasiyası (ən əsas istifadə)**
"Mənə 8 rəqəmi yarat" deyirsiniz, model yeni, əvvəl heç görünməyən bir "8" şəkli çəkir. Klassik model (classifier) şəkli görüb "bu nədir?" deyir; CVAE isə tərsini edir: "bu olsun" deyirsiniz, şəkli yaradır.

**2. Data augmentation**
Bəzi siniflərdə məlumat azdırsa (məsələn, tibbi şəkillərdə nadir xəstəlik), CVAE ilə həmin sinif üçün süni, lakin real görünən nümunələr yaradıb təlim verisini artırırsınız. Classifier sonra daha yaxşı öyrənir.

**3. Üslubu sinifdən ayırmaq (disentanglement)**
Sizin 9×9 şəbəkə təcrübəsi məhz bunu yoxlayır: latent fəza yalnız üslubu (qalınlıq, əyiklik) saxlayır, sinif isə label-dən gəlir. Bu, modelin nəyi öyrəndiyini anlamaq üçün vacibdir.

**4. Üslub transferi**
Bir şəkildən latent kod (üslub) çıxarırsınız, label-i dəyişib decoder-ə verirsiniz: eyni əl yazısı üslubunda başqa rəqəm alınır. Məsələn, "mənim əl yazımla 2-ni 8 et".

**5. Real tətbiqlər**
Mətndən/şərtdən şəkil generasiyası (şərti generasiya) məhz bu ideyanın böyüdülmüş formasıdır: şərt (label, mətn) verilir, model məzmun yaradır. Dərman molekulu dizaynı, musiqi generasiyası, sintetik verilər də belədir.

## Sizin təcrübənin məqsədi

Bu konkret kod praktik tətbiq deyil, **modelin düzgün öyrənib-öyrənmədiyini yoxlamaq üçün vizual testdir**. Əgər label dəyişəndə rəqəm dəyişir, latent nöqtə dəyişəndə yalnız üslub dəyişirsə, CVAE işləyir deməkdir.

Qısası: decoder-in işi yeni məlumat **yaratmaqdır**, tanımaq yox. Label-i özümüz veririk, çünki "nə yaratmaq istəyirik" sualının cavabı bizdədir.